# Calculate daily and yearly heating degree days with HOSTRADA/CERRA data

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from hostrada4py import hostradaPoint as hp
import os
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"

In [7]:
from hostrada4py import providerUI as provider_ui

provider_selector = provider_ui.create_provider_selector(
    globals(),
    initial="dwd",
    title="Select weather data provider",
    show=True,
)

def _sync_notebook_provider(change=None):
    supported = {value for _, value in provider_ui.variable_options(provider_dropdown.value)}

    current_variable = globals().get("HOSTRADA_VAR")
    if current_variable is not None and current_variable not in supported:
        globals()["HOSTRADA_VAR"] = "tas" if "tas" in supported else next(iter(supported))

    selector = globals().get("variable_selector")
    if selector is not None and hasattr(selector, "options"):
        all_options = globals().get("_hostrada_all_variable_options")
        if all_options is None:
            all_options = list(selector.options)
            globals()["_hostrada_all_variable_options"] = all_options
        filtered = [
            option for option in all_options
            if (option[1] if isinstance(option, tuple) else option) in supported
        ]
        old_values = tuple(value for value in selector.value if value in supported)
        selector.value = ()
        selector.options = filtered
        available = [option[1] if isinstance(option, tuple) else option for option in filtered]
        selector.value = old_values or (("tas",) if "tas" in available else tuple(available[:1]))

provider_dropdown.observe(_sync_notebook_provider, names="value")
_sync_notebook_provider()

## Definition of the location, the time period and download of the HOSTRADA/CERRA values

In [5]:
# DWD reference weather station Potsdam (stations-ID 03987)
lon = 13.0622
lat = 52.3812
start_UTC = "2019-01-01T00:00"
end_UTC = "2019-12-31T23:00"
df = hp.extract_values_for_point(var="tas", lon=lon, lat=lat, start=start_UTC, end=end_UTC)
fn = "HOSTRADA_tas.csv"
df.to_csv(fn, index=False)
print(f"{len(df)} rows written to {fn}.")

HTTP-range subset: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/air_temperature_mean/tas_1hr_HOSTRADA-v1-0_BE_gn_2019010100-2019013123.nc
24 rows written to HOSTRADA_tas.csv.


## Yearly heating degree days

In [ ]:
df = pd.read_csv(fn)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")
daily_mean = df.resample("D", on="time")["tas"].mean()
heatingDegreeDays = []
for i in range(len(daily_mean)):
    if daily_mean.iloc[i] < 15.0:
        heatingDegreeDays.append(20.0-daily_mean.iloc[i])
    else:
        heatingDegreeDays.append(0)
yearlyHeatingDegreeDays = sum(heatingDegreeDays)
print("Yearly heating degree days: ", yearlyHeatingDegreeDays)